# Name That LLM (simple version)

**Goal:** given one passage of text, guess which of 18 LLMs wrote it.

**The catch:** the training passages come from about 150 prompts, and every model answered every prompt once.
The test set uses *new* prompts, so the topic can't help us. Only writing style can.

**Plan**
1. Find which passages answer the same prompt (cluster by topic)
2. Turn each passage into style numbers
3. Subtract each prompt's average, which removes the topic and keeps the style
4. Train a classifier and test it on prompts it has never seen
5. Use the rule "each model answers each prompt once" to clean up the guesses

*This notebook is an explanation. It shows the method with small, readable code and does not connect to anything else:
it does not train the production model, and it has nothing to do with Airflow, Docker or the model registry. The real,
automated pipeline lives in `src/name_that_llm/` (see the README).*

In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, balanced_accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, normalize

TRAIN_FILE = "../data/train.csv"

## 1. Load the data

In [2]:
df = pd.read_csv(TRAIN_FILE).dropna(subset=["text", "model"]).reset_index(drop=True)

n_models = df["model"].nunique()
n_prompts = round(len(df) / n_models)   # each model answers each prompt once

print(len(df), "passages,", n_models, "models, about", n_prompts, "prompts")

2699 passages, 18 models, about 150 prompts


## 2. Find the hidden prompt groups

`train.csv` has a `prompt_id` column, but the test set won't. So we rebuild the groups ourselves:
turn each passage into topic numbers, then cluster. We keep only rare content words (dropping common ones)
so that passages cluster by *subject*, not by author.

In [3]:
# word weights (TF-IDF) using only rare, topical words
vectorizer = TfidfVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    sublinear_tf=True,
    min_df=2,
    max_df=0.05,
)
word_weights = vectorizer.fit_transform(df["text"])

# squash to 200 numbers per passage (this step is called LSA)
topic_vectors = TruncatedSVD(200, random_state=42).fit_transform(word_weights)
topic_vectors = normalize(topic_vectors)

# merge the closest passages together until we have one cluster per prompt
clusterer = AgglomerativeClustering(n_clusters=n_prompts, metric="cosine", linkage="average")
df["group"] = clusterer.fit_predict(topic_vectors)

**Did it work?** A good cluster has about 18 passages and never the same model twice.
Since this file has the real `prompt_id`, we can also compare against the truth.

In [4]:
sizes = df.groupby("group").size()
has_repeat = df.groupby("group")["model"].apply(lambda models: models.duplicated().any())

print("median cluster size:", sizes.median())
print("clusters with a repeated model: %.1f%%" % (100 * has_repeat.mean()))
print("agreement with real prompts (1.0 = perfect):", round(adjusted_rand_score(df["prompt_id"], df["group"]), 3))

median cluster size: 18.0
clusters with a repeated model: 2.7%
agreement with real prompts (1.0 = perfect): 0.977


## 3. Why we must split by prompt

If we split passages at random, the other 17 answers to the same prompt end up in training with *other* labels,
which confuses the model. Splitting by prompt copies the real test situation.
Here is the same simple model scored both ways.

In [5]:
simple_model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=100_000),
    LogisticRegression(C=5, class_weight="balanced", max_iter=1000),
)

random_split = StratifiedKFold(5, shuffle=True, random_state=42)
random_pred = cross_val_predict(simple_model, df["text"], df["model"], cv=random_split, n_jobs=-1)

prompt_split = GroupKFold(5)
prompt_pred = cross_val_predict(simple_model, df["text"], df["model"], cv=prompt_split, groups=df["group"], n_jobs=-1)

print("random split :", round(balanced_accuracy_score(df["model"], random_pred), 3))
print("split by prompt:", round(balanced_accuracy_score(df["model"], prompt_pred), 3), "(this is the honest number)")
print("guessing at random would give", round(1 / n_models, 3))

random split : 0.114
split by prompt: 0.313 (this is the honest number)
guessing at random would give 0.056


## 4. Style features

Three views of how a passage is written:

* **characters** (2 to 5 letters in a row): spelling and punctuation habits
* **words** (single words and pairs): phrasing
* **skeleton**: keep the 150 most common words and punctuation, replace every other word with `W`.
  `The cat sat; it was warm.` becomes `the W W ; it was W .` This captures sentence rhythm and has almost no topic in it.

In [6]:
def make_skeleton(text, common_words):
    pieces = []
    for token in re.findall(r"[A-Za-z']+|[^\s A-Za-z']", text):
        lower = token.lower()
        if lower in common_words:
            pieces.append(lower)
        elif token[0].isalpha() or token[0] == "'":
            pieces.append("W")
        else:
            pieces.append(token)   # punctuation stays as it is
    return " ".join(pieces)


def fit_features(train_text, use_skeleton=True):
    # Everything is learned from the training text only.
    all_words = [w.lower() for t in train_text for w in re.findall(r"[A-Za-z']+", t)]
    common_words = {word for word, count in Counter(all_words).most_common(150)}

    vectorizers = {
        "chars": TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=20000, dtype=np.float32),
        "words": TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=15000, dtype=np.float32),
    }
    if use_skeleton:
        vectorizers["skeleton"] = TfidfVectorizer(ngram_range=(1, 4), token_pattern=r"\S+", lowercase=False, sublinear_tf=True,
                                                  min_df=3, max_features=20000, dtype=np.float32)

    for name, vec in vectorizers.items():
        vec.fit(skeletons(train_text, common_words) if name == "skeleton" else train_text)
    return {"common_words": common_words, "vectorizers": vectorizers}


def skeletons(texts, common_words):
    return [make_skeleton(t, common_words) for t in texts]


def transform_features(features, texts):
    blocks = []
    for name, vec in features["vectorizers"].items():
        docs = skeletons(texts, features["common_words"]) if name == "skeleton" else texts
        blocks.append(vec.transform(docs).toarray())
    return np.hstack(blocks)


def build_features(train_text, test_text, use_skeleton=True):
    features = fit_features(train_text, use_skeleton)
    return transform_features(features, train_text), transform_features(features, test_text)

## 5. Remove the topic, then fix the guesses

**Centering:** inside each prompt group all 18 answers share the topic. Subtracting the group's average
cancels it out and leaves what is different about each author.

**Assignment:** each model answers each prompt exactly once, so inside a group the 18 passages should get 18 *different*
models. `linear_sum_assignment` finds the best one-to-one matching using the classifier's probabilities.

In [7]:
def center_by_group(features, groups):
    centered = features.copy()
    for g in np.unique(groups):
        rows = (groups == g)
        centered[rows] = features[rows] - features[rows].mean(axis=0)
    return centered


def assign_one_model_each(probs, groups):
    best_guess = probs.argmax(axis=1)
    for g in np.unique(groups):
        rows = np.where(groups == g)[0]
        cost = -np.log(probs[rows] + 1e-9)          # low cost = likely
        if len(rows) > n_models:                     # a merged cluster: let models repeat
            cost = np.tile(cost, int(np.ceil(len(rows) / n_models)))
        row_ids, model_ids = linear_sum_assignment(cost)
        best_guess[rows[row_ids]] = model_ids % n_models
    return best_guess

## 6. Put it together and test it

Every score below comes from prompts the model has never seen (5 folds).
We run it twice, without and with the skeleton features, so you can see what they add.
This cell takes a few minutes.

In [8]:
encoder = LabelEncoder().fit(df["model"])
y = encoder.transform(df["model"])
groups = df["group"].to_numpy()

scores = {}   # (features, step) -> list of scores, one per fold

for use_skeleton in [False, True]:
    name = "with skeleton" if use_skeleton else "without skeleton"

    for fold, (train_rows, test_rows) in enumerate(GroupKFold(5).split(df, y, groups)):
        train_x, test_x = build_features(df["text"].iloc[train_rows], df["text"].iloc[test_rows], use_skeleton)

        train_x = center_by_group(train_x, groups[train_rows])
        test_x = center_by_group(test_x, groups[test_rows])

        classifier = LogisticRegression(C=5, class_weight="balanced", max_iter=600)
        classifier.fit(train_x, y[train_rows])
        probs = classifier.predict_proba(test_x)

        plain_guess = probs.argmax(axis=1)
        fixed_guess = assign_one_model_each(probs, groups[test_rows])

        scores.setdefault((name, "centered"), []).append(balanced_accuracy_score(y[test_rows], plain_guess))
        scores.setdefault((name, "centered + assignment"), []).append(balanced_accuracy_score(y[test_rows], fixed_guess))
        print(name, "fold", fold + 1, "done")

results = pd.DataFrame({key: [np.mean(vals), np.std(vals)] for key, vals in scores.items()}, index=["mean", "std"]).T
results.round(3)

without skeleton fold 1 done


without skeleton fold 2 done


without skeleton fold 3 done


without skeleton fold 4 done


without skeleton fold 5 done


with skeleton fold 1 done


with skeleton fold 2 done


with skeleton fold 3 done


with skeleton fold 4 done


with skeleton fold 5 done


mean    std
without skeleton centered               0.381  0.013
                 centered + assignment  0.422  0.010
with skeleton    centered               0.431  0.013
                 centered + assignment  0.466  0.020

## 7. Ensemble: add a language model's opinion

A second, different kind of model can fix some of the TF-IDF model's mistakes.
We turn every passage into 768 numbers with DistilBERT (a small pretrained language model),
train a second classifier on those, and blend the two sets of probabilities.

The language model gets only a **small weight (10%)**. On its own it is weaker than TF-IDF, and at 30% it starts to hurt.
In our experiments 10% and 20% both helped in all 5 folds.
Those weights were picked on these same folds, so treat the gain as slightly optimistic.

Embedding the passages needs PyTorch + `sentence-transformers` and takes a few minutes on CPU.
The result is saved to `../data/embeddings/distilbert.npy`, so it only runs once.

In [9]:
from pathlib import Path
from sklearn.preprocessing import StandardScaler

EMBEDDING_FILE = Path("../data/embeddings/distilbert.npy")

if EMBEDDING_FILE.exists():
    embeddings = np.load(EMBEDDING_FILE)
else:
    from sentence_transformers import SentenceTransformer
    language_model = SentenceTransformer("distilbert-base-uncased-finetuned-sst-2-english", device="cpu")
    language_model.max_seq_length = 512
    embeddings = language_model.encode(df["text"].tolist(), batch_size=16, normalize_embeddings=True)
    EMBEDDING_FILE.parent.mkdir(exist_ok=True)
    np.save(EMBEDDING_FILE, embeddings)

print(embeddings.shape)

(2699, 768)


In [10]:
def blend(probs_main, probs_other, other_weight):
    # weighted geometric mean: average the log-probabilities, then turn them back into probabilities
    logs = (1 - other_weight) * np.log(probs_main + 1e-9) + other_weight * np.log(probs_other + 1e-9)
    mixed = np.exp(logs)
    return mixed / mixed.sum(axis=1, keepdims=True)


LANGUAGE_MODEL_WEIGHT = 0.1
ensemble_scores = {"TF-IDF only": [], "TF-IDF + 10% DistilBERT": []}

for fold, (train_rows, test_rows) in enumerate(GroupKFold(5).split(df, y, groups)):
    # model 1: the TF-IDF style features (C=1 worked a bit better than 5)
    train_x, test_x = build_features(df["text"].iloc[train_rows], df["text"].iloc[test_rows])
    train_x = center_by_group(train_x, groups[train_rows])
    test_x = center_by_group(test_x, groups[test_rows])
    tfidf_probs = LogisticRegression(C=1, class_weight="balanced", max_iter=600).fit(train_x, y[train_rows]).predict_proba(test_x)

    # model 2: DistilBERT numbers, centred by prompt and standardised
    train_e = center_by_group(embeddings[train_rows], groups[train_rows])
    test_e = center_by_group(embeddings[test_rows], groups[test_rows])
    scaler = StandardScaler().fit(train_e)
    lm_probs = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000).fit(scaler.transform(train_e), y[train_rows]).predict_proba(scaler.transform(test_e))

    mixed_probs = blend(tfidf_probs, lm_probs, LANGUAGE_MODEL_WEIGHT)
    for name, probs in [("TF-IDF only", tfidf_probs), ("TF-IDF + 10% DistilBERT", mixed_probs)]:
        guess = assign_one_model_each(probs, groups[test_rows])
        ensemble_scores[name].append(balanced_accuracy_score(y[test_rows], guess))
    print("fold", fold + 1, "done")

pd.DataFrame({k: [np.mean(v), np.std(v)] for k, v in ensemble_scores.items()}, index=["mean", "std"]).T.round(3)

fold 1 done


fold 2 done


fold 3 done


fold 4 done


fold 5 done


,mean,std
TF-IDF only,0.477,0.025
TF-IDF + 10% DistilBERT,0.521,0.018


## 8. Train the final model and save it as a version

So far we only *measured* how well the method works. Now we train on **all** the passages and save the result as a
numbered version, so it can be found, compared and restored later.

* The model file goes in `../models/model.joblib`. It holds only standard scikit-learn objects and plain Python, so it
  loads without this notebook.
* `../models/model_info.json` records the version, the cross-validated score from section 7, the settings, and a hash of
  `train.csv`, so every model says which data it was trained on.
* **Version numbers** look like `0.1.0`. Raise `MODEL_VERSION` when the method changes (new features, a different classifier);
  a plain retrain on the same method can just raise the last number.
* The files are then versioned with DVC and git tags (the commands are at the end of this section).

In [11]:
import datetime
import hashlib
import json

import joblib

MODEL_VERSION = "0.1.0"
MODEL_DIR = Path("../models")

In [12]:
# Train on everything, with the same settings that were scored in section 7.
features = fit_features(df["text"])
style_x = center_by_group(transform_features(features, df["text"]), groups)
style_classifier = LogisticRegression(C=1, class_weight="balanced", max_iter=600).fit(style_x, y)

embedding_x = center_by_group(embeddings, groups)
embedding_scaler = StandardScaler().fit(embedding_x)
embedding_classifier = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000).fit(
    embedding_scaler.transform(embedding_x), y)

model = {
    "classes": encoder.classes_,
    "features": features,
    "style_classifier": style_classifier,
    "embedding_scaler": embedding_scaler,
    "embedding_classifier": embedding_classifier,
    "embedding_weight": LANGUAGE_MODEL_WEIGHT,
}
print("trained on", len(df), "passages")

trained on 2699 passages


In [13]:
MODEL_DIR.mkdir(exist_ok=True)
info_file = MODEL_DIR / "model_info.json"
previous = json.loads(info_file.read_text()) if info_file.exists() else None

info = {
    "version": MODEL_VERSION,
    "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "balanced_accuracy": round(float(np.mean(ensemble_scores["TF-IDF + 10% DistilBERT"])), 3),
    "how_scored": "5-fold cross-validation, whole prompts held out, centering + assignment (section 7)",
    "settings": {"style_c": 1, "embedding_c": 0.1, "embedding_weight": LANGUAGE_MODEL_WEIGHT},
    "training_passages": len(df),
    "train_csv_md5": hashlib.md5(Path(TRAIN_FILE).read_bytes()).hexdigest(),   # the same hash DVC records for train.csv
}

if previous is not None:
    print("previous version:", previous["version"], "scored", previous["balanced_accuracy"])
    print("this version    :", info["version"], "scored", info["balanced_accuracy"])
    if previous["version"] == info["version"]:
        print("same version number as before: raise MODEL_VERSION if the method changed")

joblib.dump(model, MODEL_DIR / "model.joblib")
info_file.write_text(json.dumps(info, indent=2))
print("saved", MODEL_DIR / "model.joblib", "and", info_file)
info

saved ../models/model.joblib and ../models/model_info.json


{'version': '0.1.0',
 'trained_at': '2026-11-08T16:13:22',
 'balanced_accuracy': 0.521,
 'how_scored': '5-fold cross-validation, whole prompts held out, centering + assignment (section 7)',
 'settings': {'style_c': 1, 'embedding_c': 0.1, 'embedding_weight': 0.1},
 'training_passages': 2699,
 'train_csv_md5': '61f581c1bdd4664d588a7c23c413ba0a'}

### Check that the saved model works

Load the file the way someone else would, and use it to predict. Passages have to be grouped by prompt first, because
centering and assignment need the other passages from the same prompt; here we use the groups from section 2.
These 10 prompts were part of the training data, so the accuracy only proves the saved model loads and runs. The honest
score is the one in section 7.

In [14]:
def predict_with(saved, texts, embeddings, groups):
    style_x = center_by_group(transform_features(saved["features"], texts), groups)
    style_probs = saved["style_classifier"].predict_proba(style_x)

    embedding_x = saved["embedding_scaler"].transform(center_by_group(embeddings, groups))
    embedding_probs = saved["embedding_classifier"].predict_proba(embedding_x)

    probs = blend(style_probs, embedding_probs, saved["embedding_weight"])
    return saved["classes"][assign_one_model_each(probs, groups)]


loaded = joblib.load(MODEL_DIR / "model.joblib")

ten_prompts = sorted(df["prompt_id"].unique())[:10]
rows = df.index[df["prompt_id"].isin(ten_prompts)]

guesses = predict_with(loaded, df["text"][rows], embeddings[rows], groups[rows])
print(len(rows), "passages, accuracy on these training prompts:", round((guesses == df["model"][rows].to_numpy()).mean(), 3))

179 passages, accuracy on these training prompts: 1.0


### Version the data and the model

The notebook has now written `../models/model.joblib` and `../models/model_info.json`. To record this as a version, run
these in a terminal from the project folder (DVC stores the big file by its hash; git keeps the small pointer and the info):

```
dvc add models/model.joblib
git add models/model.joblib.dvc models/model_info.json models/.gitignore
git commit -m "Model v0.1.0"
git tag model-v0.1.0
dvc push
```

To go back to an older version later: `git checkout model-v0.1.0 -- models/model.joblib.dvc models/model_info.json`, then
`dvc checkout models/model.joblib`. The training data is versioned the same way (`dvc add data/train.csv`, tag `data-v2`, ...).

## What to take away

* Splitting by prompt gives the honest score. Random splits mislead.
* Centering and assignment both help a lot, because both use the fact that each prompt has exactly one answer per model.
* The skeleton features add a further gain.
* Models from the same company (for example the Gemini ones) get mixed up with each other most.
* Blending in a small share of a language model (DistilBERT) adds a further gain; averaging more TF-IDF models does not.
* At test time the same recipe needs the test passages grouped by prompt too.
* Data and models are versioned (DVC and git tags), and each saved model records its version, its score and the hash of the data it was trained on.